<a href="https://colab.research.google.com/github/mybenkhadda/GEMS---drivendata/blob/main/notebooks/colab/GEMS_FAULT_DISCOVERY_RESET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEMS fault-discovery reset (code rev fdr.1-r1)

**Scientific diagnosis -> visual geological interpretation -> independent evidence -> minimal benchmark -> conditional challenger -> validated submission.** Not a master pipeline: it reuses the submission factory (frozen B0 / B1 / W2 engine), the GeoMoE scenarios / scorer / stage machinery, the M7 DEM pipeline and the factory GeoTIFF writer.

| mode | stages |
|---|---|
| `ATLAS` (start here) | tests, attach, audit, scenarios, atlas, physics |
| `BENCHMARK` | + representation / baseline benchmark vs the frozen submitted W2 |
| `DEM_PILOT` | tests, attach, audit, scenarios, terrain pilot (only with `dem.enabled`) |
| `CHALLENGER` | everything up to the evidence-based challenger decision |
| `FULL` / `RESUME` | + export (factory writer; manual upload only) |

Nothing trains a neural network. DEM downloads happen only with `dem.enabled` **and** `dem.allow_download`. A failed / blocked upstream stage gives `BLOCKED_UPSTREAM` or `NOT_EVALUATED`, never a conclusion. Outputs: `MyDrive/GEMS_2/outputs/fault_discovery_reset/`.

In [ ]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'main',                         # pin the pushed commit SHA
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',

    'mode': 'ATLAS',                       # ATLAS | BENCHMARK | DEM_PILOT | CHALLENGER | FULL | RESUME
    'reuse_cached': True,
    'native_resolution_m': 100,

    'atlas': {'n_examples': 48, 'seed': 2026, 'patch_size_px': 256},
    'dem': {'enabled': False, 'cached_only': True, 'pilot_only': True,
            'process_cached_tiles': False,  # True: process pilot tiles already on disk / Drive
            'allow_download': False},       # True: M7's storage-checked pilot download (opt-in)
    'benchmark': {},                       # overrides of configs/reset/fault_discovery_reset.yaml 'benchmark'
    'challenger': {'enabled': False, 'train_large_neural_model': False},
    'export': {'enabled': True, 'manual_upload_only': True, 'exploratory': False},

    'run_synthetic_tests': True,
    'save_all_artifacts': True,
    'sync_to_drive': True,
}
PRIVATE_REPO = False
EXPECT = {'code_rev': 'fdr.1-r1', 'commit': None}

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = 'https://github.com/' + CONFIG['repo'] + '.git'

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    hdr = base64.b64encode(('x-access-token:' + (userdata.get('GITHUB_TOKEN') or '')).encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(args[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r

if not Path(CODE, '.git').exists():
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + ref, check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'reset_colab_active', 'origin/' + ref)
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: Runtime -> Restart session, then run from the top')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab]', 'pytest'], check=True)
sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD

import pandas as pd
from IPython.display import HTML, Image, Markdown, display
import gems_bench.reset as RS
from gems_bench.reset import run as RR
from gems_bench.geomoe import run as GR
from gems_bench.leaderboard_focus import factory as FA
assert RS.RESET_CODE_REV == EXPECT['code_rev'], f'code rev {RS.RESET_CODE_REV} != {EXPECT["code_rev"]}'
assert not EXPECT['commit'] or HEAD.startswith(EXPECT['commit']), f'HEAD {HEAD} != {EXPECT["commit"]}'
RNB = RR.reset_config(CONFIG)
F = FA.Factory({**GR.factory_config({**GR.NB_DEFAULTS, **{k: CONFIG[k] for k in ('repo', 'ref', 'drive_root', 'local_root', 'reuse_cached', 'sync_to_drive')}})}, HEAD)
R = RR.FaultDiscoveryReset(CONFIG, HEAD, factory=F)
OUT = R.ws.root
STAGES = RR.MODES[RNB['mode']]
RUN_ALL = R.run

def show(rel, cols=None, n=40):
    p = OUT / rel
    if not p.exists():
        print('ABSENT:', rel); return None
    df = pd.read_csv(p) if p.suffix == '.csv' else pd.read_parquet(p)
    display(df[[c for c in (cols or df.columns) if c in df]].head(n)); return df

print('SHA', HEAD, '| rev', RS.RESET_CODE_REV, '| mode', RNB['mode'], '| stages', STAGES)
print('Drive outputs:', R.drive / R.out)

## [00] Synthetic tests (enforced) -> [01] frozen engine -> [02] scientific audit -> [03] validation scenarios

A failing test (`TESTS_FAILED`) blocks the atlas, benchmark, terrain and export. The audit classifies every historical claim (verified / reproducible / questionable / hypothesis / not available) and runs its Colab check (e.g. the malformed EDA transform labels, the saturated evidence).

In [ ]:
M = R.s00_tests()
if R.status['00_tests'] == 'TESTS_FAILED':
    print((OUT / 'audit/pytest_output.txt').read_text()[-4000:])
R.s01_attach(); R.s02_reset_audit(); R.s03_scenarios()
show('audit/prior_results_audit.csv', ['finding_id', 'classification', 'check_result', 'claim', 'check_detail'])
show('audit/source_inventory.csv')

## [04] Geological evidence atlas

Categories A-H on honest (zone-stitched held-out) maps, seeded, round-robin over regions, minimum separation. Open `atlas/atlas_overview.html` on Drive (or below). **Review EVIDENCE_ONLY panels first** (no labels, no model outputs); DIAGNOSTIC panels reveal mapped faults and B0 / B1 / W2.

In [ ]:
if '04_atlas' in STAGES:
    R.s04_atlas()
    C = show('atlas/atlas_index.csv', ['candidate_id', 'category', 'region_id', 'review_split', 'x_utm', 'y_utm', 'distance_to_known_fault_km'], n=60)
    if C is not None and len(C):
        display(Image(str(OUT / f'atlas/panels/{C.candidate_id.iloc[0]}_evidence_only.png')))

### Review (append-only)

Either use the widget, or fill `atlas/annotation_template.csv` (copy it, edit on Drive) and import it. Every save appends a row to `atlas/reviewed_annotations.csv` (never overwritten). `reviewer` is required; `review_mode` = EVIDENCE_ONLY | DIAGNOSTIC. A review is *independent* only when made in EVIDENCE_ONLY mode citing no mapped-fault / model source.

In [ ]:
from gems_bench.reset.atlas import review_widget
W = review_widget(R)
if W is not None: display(W)
# Alternative: R.import_reviews('/content/drive/MyDrive/GEMS_2/outputs/fault_discovery_reset/atlas/my_reviews.csv')
# One review in code: R.record_review({'candidate_id': '...', 'interpretation': 'POSSIBLE_FAULT', 'confidence': 'MEDIUM',
#                                      'review_mode': 'EVIDENCE_ONLY', 'reviewer': 'name', 'evidence_sources': 'magnetic;gravity',
#                                      'supporting_evidence': '...', 'alternative_explanation': '...', 'data_license': 'n/a'})

## [05] Physical evidence (contrast vs disruption vs continuity; scale and localisation)

In [ ]:
if '05_physics' in STAGES:
    R.s05_physics()
    show('physics/localization_error.csv')
    E = show('physics/evidence_by_modality.csv', n=0)
    if E is not None: display(E.groupby(['modality', 'status']).size().unstack(fill_value=0))

## [06] Benchmark: frozen submitted W2 (A4) vs representations (A5) and label-free / CNN baselines (A1-A3)

Same units, eligible masks, scorer; realised area and mass recorded; A5 / A1-A3 use the fixed-policy rule (development units only).

In [ ]:
if '06_benchmark' in STAGES:
    R.s06_benchmark()
    show('benchmark/baseline_metrics.csv', ['scenario', 'model', 'protocol', 'dti', 'worst_zone_dti', 'params_all'])
    show('benchmark/paired_comparison.csv')
    print(json.dumps({k: (R.ws.read_json('benchmark/selected_policy.json') or {}).get(k) for k in ('status', 'deployable_cell', 'exportable', 'rule')}, indent=1, default=str))

## [07] DEM pilot (opt-in) -> [08] challenger decision -> [09] export

In [ ]:
if '07_terrain' in STAGES:
    R.s07_terrain(); print(json.dumps(R.ws.read_json('terrain/pilot_decision.json'), indent=1, default=str)[:1500])
if '08_challenger' in STAGES:
    R.s08_challenger(); print(json.dumps(R.ws.read_json('challenger/selected_challenger.json'), indent=1, default=str)[:2000])
if '09_export' in STAGES:
    R.s09_export(); show('submissions/submission_manifest.csv')

## [11] Stage integrity, [12] reports and figures (always)

In [ ]:
R.s11_integrity(); R.s12_reports()
show('audit/stage_integrity.csv', ['stage', 'actual_status', 'missing_outputs', 'failure_reason', 'affected_file', 'scientific_conclusion_allowed'])
for f in ('geological_atlas_overview', 'scale_analysis', 'localization_analysis', 'dti_vs_budget', 'binary_vs_continuous', 'model_comparison'):
    p = OUT / 'figures' / f'{f}.png'
    if p.exists(): display(Image(str(p)))
for r in ('SCIENTIFIC_AUDIT', 'CHALLENGER_DECISION', 'LEADERBOARD_RECOMMENDATION', 'NEXT_ACTION'):
    p = OUT / f'reports/{r}.md'
    if p.exists(): display(Markdown(p.read_text()))